In [1]:
from tavily import TavilyClient
from langchain.tools import tool
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def tavily_search(query: str) -> str:
    """Search the web using Tavily."""
    
    results = tavily.search(
        query=query,
        max_results=5
    )

    output = []

    for item in results["results"]:
        output.append(
            f"Title: {item['title']}\n"
            f"URL: {item['url']}\n"
            f"Content: {item['content']}"
        )

    return "\n\n".join(output)

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

In [3]:
researcher_subagent = {
    "name": "researcher",
    "description": "Researches topics using Tavily web search and returns detailed findings.",
    "system_prompt": """
You are a professional research agent.

Always:
1. Search the web first.
2. Gather multiple sources.
3. Summarize findings.
4. Return a structured report.
""",
    "tools": [tavily_search],
    "model": llm
}

In [5]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are a supervisor agent.

Delegate any research-related task to the researcher subagent.
""",
    subagents=[researcher_subagent]
)

In [12]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """LLM Gateways?

Return the answer in this JSON format:

{
  "topic": "",
  "summary": "",
  "key_findings": [],
  "sources": []
}
"""
            }
        ]
    }
)

In [13]:
print(type(result))
print(result)

<class 'dict'>
{'messages': [HumanMessage(content='LLM Gateways?\n\nReturn the answer in this JSON format:\n\n{\n  "topic": "",\n  "summary": "",\n  "key_findings": [],\n  "sources": []\n}\n', additional_kwargs={}, response_metadata={}, id='11df1a32-bb7b-406e-9a6f-eb5d125574a8'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T17:33:48.0757027Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7250418100, 'load_duration': 113414800, 'prompt_eval_count': 4096, 'prompt_eval_duration': 3170300600, 'eval_count': 94, 'eval_duration': 3816976800, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019ead72-7858-75f3-a1dd-9cf5471ad66e-0', tool_calls=[{'name': 'task', 'args': {'topic': 'LLM Gateways', 'summary': 'LLM Gateways are a type of subagent that can be used to handle complex, multi-step independent tasks with isolated context windows.', 'key_findings': ['LLM Gateways can be used 